# RescueVision Edge — Training Notebook
**Hackathon FindIT! 2026 — Track A: The Edge Vision**

Model: YOLOv8n (Ultralytics) → ONNX export  
Dataset: VisDrone-DET 2019, kelas pedestrian only  

> **PENTING:** Jalankan semua cell secara berurutan. Jangan clear output sebelum submit — log training adalah bukti constraint compliance.

## 0. Environment Check

In [1]:
import sys, os, platform
import torch

print(f'Python     : {sys.version}')
print(f'PyTorch    : {torch.__version__}')
print(f'CUDA       : {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"})')
print(f'Platform   : {platform.system()} {platform.machine()}')
print(f'Working dir: {os.getcwd()}')

# Pastikan working dir adalah root repo
if not os.path.exists('dataset.yaml'):
    os.chdir('..')  # naik satu level jika dijalankan dari notebooks/
    print(f'Changed to : {os.getcwd()}')

Python     : 3.10.18 | packaged by Anaconda, Inc. | (main, Jun  5 2025, 13:08:55) [MSC v.1929 64 bit (AMD64)]
PyTorch    : 2.5.1+cu121
CUDA       : True (NVIDIA GeForce RTX 4060)
Platform   : Windows AMD64
Working dir: c:\Users\MASTER CORE\RescueVision\notebooks
Changed to : c:\Users\MASTER CORE\RescueVision


In [2]:
from ultralytics import YOLO
import ultralytics
print(f'Ultralytics: {ultralytics.__version__}')

# Cek dataset tersedia
from pathlib import Path
assert Path('train_data/images/train').exists(), 'STOP: Jalankan src/prepare_visdrone.py dulu'
assert Path('test_data/images').exists(), 'STOP: test_data belum ada'

train_imgs = list(Path('train_data/images/train').glob('*.jpg'))
val_imgs   = list(Path('train_data/images/val').glob('*.jpg'))
test_imgs  = list(Path('test_data/images').glob('*.jpg'))
print(f'Train images: {len(train_imgs)}')
print(f'Val images  : {len(val_imgs)}')
print(f'Test images : {len(test_imgs)}')

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\MASTER CORE\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics: 8.4.31
Train images: 5655
Val images  : 530
Test images : 1265


## 1. Dataset Verification (Data Leakage Check)

In [3]:
# WAJIB: Buktikan tidak ada overlap antara train dan test
train_names = set(p.stem for p in Path('train_data/images/train').glob('*.jpg'))
val_names   = set(p.stem for p in Path('train_data/images/val').glob('*.jpg'))
test_names  = set(p.stem for p in Path('test_data/images').glob('*.jpg'))

overlap_train_test = train_names & test_names
overlap_val_test   = val_names & test_names
overlap_train_val  = train_names & val_names

print('=== Data Leakage Validation ===')
print(f'Train ∩ Test : {len(overlap_train_test)} files  ← harus 0')
print(f'Val   ∩ Test : {len(overlap_val_test)} files  ← harus 0')
print(f'Train ∩ Val  : {len(overlap_train_val)} files  ← harus 0')

assert len(overlap_train_test) == 0, f'DATA LEAKAGE DETECTED: {overlap_train_test}'
assert len(overlap_val_test)   == 0, f'DATA LEAKAGE DETECTED: {overlap_val_test}'
assert len(overlap_train_val)  == 0, f'DATA LEAKAGE DETECTED: {overlap_train_val}'
print('✅ No data leakage detected — constraint compliance OK')

=== Data Leakage Validation ===
Train ∩ Test : 0 files  ← harus 0
Val   ∩ Test : 0 files  ← harus 0
Train ∩ Val  : 0 files  ← harus 0
✅ No data leakage detected — constraint compliance OK


## 2. Training Configuration

In [4]:
# Konfigurasi training — tuning untuk VisDrone small object
TRAIN_CONFIG = {
    'data'       : 'dataset.yaml',
    'epochs'     : 100,           # Minimum. Naikkan ke 150+ jika waktu cukup
    'imgsz'      : 640,           # Input resolution. VisDrone butuh resolusi tinggi
    'batch'      : 16,            # Sesuaikan dengan VRAM GPU lab
    'device'     : 0,             # GPU index. Ganti ke 'cpu' jika tidak ada GPU
    'workers'    : 4,
    'patience'   : 20,            # Early stopping
    'save'       : True,
    'save_period': 10,            # Save checkpoint setiap 10 epoch
    'project'    : 'runs/train',
    'name'       : 'rescuevision_v1',
    
    # Augmentasi — penting untuk generalisasi di kondisi bencana
    'hsv_h'      : 0.015,         # Hue jitter
    'hsv_s'      : 0.7,           # Saturation jitter (asap, debu)
    'hsv_v'      : 0.4,           # Value jitter (pencahayaan tidak merata)
    'flipud'     : 0.0,           # Vertical flip (drone view — jarang dibalik)
    'fliplr'     : 0.5,           # Horizontal flip
    'scale'      : 0.5,           # Scale jitter (variasi ketinggian drone)
    'mosaic'     : 1.0,           # Mosaic augmentation (efektif untuk small object)
    'mixup'      : 0.1,
    
    # Loss weights — prioritaskan recall untuk search & rescue
    'box'        : 7.5,
    'cls'        : 0.5,
    'dfl'        : 1.5,
}

print('Training config:')
for k, v in TRAIN_CONFIG.items():
    print(f'  {k:12s}: {v}')

Training config:
  data        : dataset.yaml
  epochs      : 100
  imgsz       : 640
  batch       : 16
  device      : 0
  workers     : 4
  patience    : 20
  save        : True
  save_period : 10
  project     : runs/train
  name        : rescuevision_v1
  hsv_h       : 0.015
  hsv_s       : 0.7
  hsv_v       : 0.4
  flipud      : 0.0
  fliplr      : 0.5
  scale       : 0.5
  mosaic      : 1.0
  mixup       : 0.1
  box         : 7.5
  cls         : 0.5
  dfl         : 1.5


## 3. Model Initialization

In [5]:
# Load YOLOv8n pretrained (COCO) — fine-tune untuk VisDrone
model = YOLO('yolov8n.pt')  # Auto-download jika belum ada
print(model.info())

YOLOv8n summary: 129 layers, 3,157,200 parameters, 0 gradients, 8.9 GFLOPs
(129, 3157200, 0, 8.8575488)


## 4. Training

In [8]:
# ============================================================
# MULAI TRAINING
# Jangan interrupt kecuali terpaksa — log ini akan menjadi
# bukti training untuk penilaian juri
# ============================================================
results = model.train(**TRAIN_CONFIG)
print('\n=== Training Complete ===')
print(f'Best model saved at: {results.save_dir}/weights/best.pt')

Ultralytics 8.4.31  Python-3.10.18 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=rescuevision_v13, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=20, perspect

## 5. Evaluation on Test Set

In [13]:
import shutil

# Load best model
best_pt_path = f'runs/detect/runs/train/rescuevision_v13/weights/best.pt'
best_model = YOLO(best_pt_path)

# Evaluate on test set
metrics = best_model.val(
    data='dataset.yaml',
    split='test',
    imgsz=640,
    device='cpu',   # Validasi di CPU sesuai kondisi juri
    batch=1,
)

print('\n=== TEST SET METRICS ===')
print(f'mAP@50    : {metrics.box.map50:.4f}')
print(f'mAP@50-95 : {metrics.box.map:.4f}')
print(f'Precision : {metrics.box.mp:.4f}')
print(f'Recall    : {metrics.box.mr:.4f}')

# Copy ke folder model/
os.makedirs('model', exist_ok=True)
shutil.copy2(best_pt_path, 'model/best.pt')
print(f'\nBest weights copied to model/best.pt')

Ultralytics 8.4.31  Python-3.10.18 torch-2.5.1+cu121 CPU (12th Gen Intel Core i5-12400F)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 44.419.7 MB/s, size: 236.0 KB)
val: Scanning C:\Users\MASTER CORE\RescueVision\test_data\labels... 1265 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1265/1265 625.0it/s 2.0s0.1s
val: New cache created: C:\Users\MASTER CORE\RescueVision\test_data\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1265/1265 26.9it/s 47.1s<0.1s
                   all       1265      25778      0.522      0.296      0.304      0.116
Speed: 0.2ms preprocess, 27.5ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to C:\Users\MASTER CORE\RescueVision\runs\detect\val

=== TEST SET METRICS ===
mAP@50    : 0.3038
mAP@50-95 : 0.1157
Precision : 0.5219
Recall    : 0.2960

Best weights copied to model/best.pt


## 6. ONNX Export (Constraint C-A2, C-A4)

In [14]:
import os

# Export ke ONNX — ini yang akan digunakan untuk inference di CPU
print('Exporting to ONNX...')
export_path = best_model.export(
    format='onnx',
    imgsz=640,
    dynamic=False,
    simplify=True,   # ONNX simplifier — kurangi ukuran & percepat inference
    opset=17,
)

# Copy ONNX ke model/
import shutil
shutil.copy2(export_path, 'model/best.onnx')

# Check size
pt_size   = os.path.getsize('model/best.pt') / (1024**2)
onnx_size = os.path.getsize('model/best.onnx') / (1024**2)
print(f'\n=== MODEL SIZE CHECK ===')
print(f'PyTorch (.pt) : {pt_size:.2f} MB')
print(f'ONNX          : {onnx_size:.2f} MB')
print(f'Limit         : 50 MB')
assert onnx_size <= 50, f'CONSTRAINT VIOLATION: ONNX model {onnx_size:.2f} MB > 50 MB'
print(f'✅ Constraint C-A1 PASS: {onnx_size:.2f} MB ≤ 50 MB')

Exporting to ONNX...
Ultralytics 8.4.31  Python-3.10.18 torch-2.5.1+cu121 CPU (12th Gen Intel Core i5-12400F)
 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from 'runs\detect\runs\train\rescuevision_v13\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (5.9 MB)
requirements: Ultralytics requirements ['onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
   ---------------------------------------- 0.0/244.5 MB ? eta -:--:--
   ---------------------------------------- 2.1/244.5 MB 10.7 MB/s eta 0:00:23
    --------------------------------------- 3.9/244.5 MB 9.8 MB/s eta 0:00:25
    --------------------------------------- 5.8/244.5 MB 9.5 MB/s eta 0:00:26
   - -------------------------------------- 7.6/244.5 MB 9.6 MB/s eta 0:00:25
   - -------------------------------------- 9.2/244.5 MB 9.2 MB/s eta 0:00:26
   - ----

## 7. Training Results Visualization

In [17]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

results_dir = Path('runs/detect/runs/train/rescuevision_v13')

# Plot training curves
results_png = results_dir / 'results.png'
if results_png.exists():
    img = mpimg.imread(str(results_png))
    plt.figure(figsize=(16, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Training Curves')
    plt.tight_layout()
    plt.show()

# PR curve
pr_png = results_dir / 'PR_curve.png'
if pr_png.exists():
    img = mpimg.imread(str(pr_png))
    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title('PR Curve')
    plt.show()

<Figure size 1600x800 with 1 Axes>

## 8. Summary
Jalankan cell ini terakhir untuk summary yang bisa dikopi ke Proposal Bab 3.

In [18]:
import time
import onnxruntime as ort
import numpy as np

# Quick CPU inference time check
session = ort.InferenceSession('model/best.onnx', providers=['CPUExecutionProvider'])
input_name = session.get_inputs()[0].name
dummy = np.random.randn(1, 3, 640, 640).astype(np.float32)

# Warmup
for _ in range(3):
    session.run(None, {input_name: dummy})

# Benchmark 20 runs
times = []
for _ in range(20):
    t = time.perf_counter()
    session.run(None, {input_name: dummy})
    times.append(time.perf_counter() - t)

mean_ms = np.mean(times) * 1000
max_ms  = np.max(times)  * 1000

print('=' * 50)
print('CONSTRAINT COMPLIANCE SUMMARY (untuk Bab 3)')
print('=' * 50)
print(f'C-A1 Model Size    : {onnx_size:.2f} MB ≤ 50 MB   ✅')
print(f'C-A2 CPU Compat    : ONNX Runtime CPU-only         ✅')
print(f'C-A3 Inference Time: {mean_ms:.1f} ms mean, {max_ms:.1f} ms max ≤ 3000 ms ✅' if max_ms <= 3000 else f'C-A3 FAIL: {max_ms:.1f} ms > 3000 ms ❌')
print(f'C-A4 Framework     : Ultralytics YOLOv8 + ONNX     ✅')
print(f'C-A5 Offline Total : No external API               ✅')
print('=' * 50)

CONSTRAINT COMPLIANCE SUMMARY (untuk Bab 3)
C-A1 Model Size    : 11.70 MB ≤ 50 MB   ✅
C-A2 CPU Compat    : ONNX Runtime CPU-only         ✅
C-A3 Inference Time: 28.0 ms mean, 31.4 ms max ≤ 3000 ms ✅
C-A4 Framework     : Ultralytics YOLOv8 + ONNX     ✅
C-A5 Offline Total : No external API               ✅
